# Cyber Threat Intelligence Pipeline
## Apache Spark + Medallion Architecture

This notebook processes NVD, EPSS, and CISA KEV feeds to score cybersecurity vulnerabilities.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType,
    TimestampType, ArrayType, BooleanType, IntegerType
)
from pyspark.sql.functions import col, from_json, to_date, regexp_extract, explode, sum as _sum

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("CyberThreatIntel") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Spark session initialized on Serverless compute")

# Base path for data files (absolute path required on Serverless compute)
DATA_BASE = "/Workspace/Users/naumaniqbal987@gmail.com/cyber-threat-intel/data/sample_raw/full_load"

# ==========================================
# SCHEMA DEFINITIONS
# ==========================================

# EPSS Schema (CSV data)
epss_schema = StructType([
    StructField("cve", StringType(), True),
    StructField("epss", DoubleType(), True),
    StructField("percentile", DoubleType(), True)
])

# CISA KEV Schema (JSON data)
kev_schema = StructType([
    StructField("cveID", StringType(), True),
    StructField("vendorProject", StringType(), True),
    StructField("product", StringType(), True),
    StructField("vulnerabilityName", StringType(), True),
    StructField("dateAdded", DateType(), True),
    StructField("shortDescription", StringType(), True),
    StructField("requiredAction", StringType(), True),
    StructField("dueDate", DateType(), True),
    StructField("knownRansomwareCampaignUse", StringType(), True),
    StructField("forensicTriage", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("cwes", ArrayType(StringType()), True)
])

# NVD CVE Schema (simplified - focusing on key fields)
nvd_schema = StructType([
    StructField("id", StringType(), True),
    StructField("sourceIdentifier", StringType(), True),
    StructField("published", TimestampType(), True),
    StructField("lastModified", TimestampType(), True),
    StructField("vulnStatus", StringType(), True),
    StructField("descriptions", ArrayType(
        StructType([
            StructField("lang", StringType(), True),
            StructField("value", StringType(), True)
        ])
    ), True),
    StructField("metrics", StringType(), True),  # Simplified as string for now
    StructField("references", StringType(), True)  # Simplified as string for now
])

# ==========================================
# DATA LOADING
# ==========================================

print("\n" + "="*50)
print("Loading EPSS Data")
print("="*50)

# Load EPSS CSV (skip comment line)
epss_df = spark.read \
    .option("header", "true") \
    .option("comment", "#") \
    .schema(epss_schema) \
    .csv(f"{DATA_BASE}/epss_full_sample.csv")

print(f"EPSS Record Count: {epss_df.count()}")
print("EPSS Schema:")
epss_df.printSchema()
print("EPSS Sample Data:")
epss_df.show(5, truncate=False)

print("\n" + "="*50)
print("Loading CISA KEV Data")
print("="*50)

# Load CISA KEV JSON
kev_df = spark.read \
    .option("multiline", "true") \
    .json(f"{DATA_BASE}/kev_full_sample.json")

# Extract vulnerabilities array directly (already typed as array of structs by Spark)
kev_vulnerabilities = kev_df.select(
    explode(col("vulnerabilities")).alias("vulnerability")
)

# Flatten the structure
kev_final_df = kev_vulnerabilities.select("vulnerability.*")

# Convert date strings to proper date types
kev_final_df = kev_final_df \
    .withColumn("dateAdded", to_date(col("dateAdded"), "yyyy-MM-dd")) \
    .withColumn("dueDate", to_date(col("dueDate"), "yyyy-MM-dd"))

print(f"KEV Record Count: {kev_final_df.count()}")
print("KEV Schema:")
kev_final_df.printSchema()
print("KEV Sample Data:")
kev_final_df.show(5, truncate=False)

print("\n" + "="*50)
print("Loading NVD CVE Data")
print("="*50)

# Load NVD CVE JSON
nvd_df = spark.read \
    .option("multiline", "true") \
    .json(f"{DATA_BASE}/nvd_cves_full_sample.json")

# Extract vulnerabilities array directly (already typed as array of structs by Spark)
nvd_vulnerabilities = nvd_df.select(
    explode(col("vulnerabilities")).alias("vulnerability")
)

# Flatten the cve structure
nvd_final_df = nvd_vulnerabilities.select("vulnerability.cve.*")

print(f"NVD Record Count: {nvd_final_df.count()}")
print("NVD Schema:")
nvd_final_df.printSchema()
print("NVD Sample Data:")
nvd_final_df.show(5, truncate=False)

# ==========================================
# DATA VALIDATION
# ==========================================

print("\n" + "="*50)
print("Data Validation Summary")
print("="*50)

print(f"EPSS DataFrame: {epss_df.count()} records")
print(f"KEV DataFrame: {kev_final_df.count()} records") 
print(f"NVD DataFrame: {nvd_final_df.count()} records")

# Check for null values in key fields
print("\nEPSS Null Checks:")
epss_df.select([col(c).isNull().cast("int").alias(c) for c in epss_df.columns]) \
    .agg(*[_sum(col(c)).alias(c) for c in epss_df.columns]) \
    .show()

print("\nKEV Null Checks:")
kev_final_df.select([col(c).isNull().cast("int").alias(c) for c in ["cveID", "dateAdded", "dueDate"]]) \
    .agg(*[_sum(col(c)).alias(c) for c in ["cveID", "dateAdded", "dueDate"]]) \
    .show()

print("\nNVD Null Checks:")
nvd_final_df.select([col(c).isNull().cast("int").alias(c) for c in ["id", "published", "lastModified"]]) \
    .agg(*[_sum(col(c)).alias(c) for c in ["id", "published", "lastModified"]]) \
    .show()

print("\n" + "="*50)
print("Data Loading Complete!")
print("="*50)

# Stop Spark session
# spark.stop()